# Compile paired prompt variants

Read deterministic alterations and Qwen rewrites into one paired prompt database. New v2 rewrite outputs are preferred; historical preparation databases remain supported.

This version preserves rewrite completion/error flags, full generation metadata, source snapshot metadata, and database hashes. It writes **answercarefully_test_prompts_v2.sqlite3**, leaving the historical compiled database unchanged. Keep source IDs and dev/test splits intact.

Run the unified `data_ready/prepare_prompt_variants.ipynb` for new Qwen preparations. A shared source ID does not certify semantic equivalence; that still requires review.


In [ ]:
from pathlib import Path

DRIVE_DIR = Path("/content/drive/MyDrive/ez-penta")
def rewrite_db(strategy):
    updated = DRIVE_DIR / f"answercarefully_qwen_{strategy}_v2.sqlite3"
    return updated if updated.is_file() else DRIVE_DIR / f"answercarefully_qwen_{strategy}.sqlite3"

SIMPLE_DB = DRIVE_DIR / "answercarefully_simple.sqlite3"
DB_OUT = DRIVE_DIR / "answercarefully_test_prompts_v2.sqlite3"
WORK_DIR = Path("/content/work")
KEY = ["source_id", "source_split"]
SHARED = ["text", "text_en", "output"]


## 1. Download the dbs from Google Drive

In [ ]:
import shutil

if not Path("/content/drive/MyDrive").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
QWEN_DBS = {strategy: rewrite_db(strategy) for strategy in ("fc", "rp", "rs")}
missing = [path.name for path in [*QWEN_DBS.values(), SIMPLE_DB] if not path.exists()]
if missing:
    raise FileNotFoundError(f"Not in {DRIVE_DIR}: {missing}. Prepare these variants first.")
WORK_DIR.mkdir(parents=True, exist_ok=True)
for path in [*QWEN_DBS.values(), SIMPLE_DB]:
    shutil.copyfile(path, WORK_DIR / path.name)
    print("Selected source:", path.name)


In [ ]:
import hashlib
import json
import sqlite3
from contextlib import closing
import pandas as pd

def read_db(path):
    local = WORK_DIR / path.name
    with closing(sqlite3.connect(local.resolve().as_uri() + "?mode=ro", uri=True)) as connection:
        records = pd.read_sql_query("SELECT * FROM records", connection)
        info = pd.read_sql_query("SELECT * FROM generation_info", connection)
        has_snapshot = connection.execute("SELECT 1 FROM sqlite_master WHERE name='snapshot_info'").fetchone()
        snapshot = pd.read_sql_query("SELECT * FROM snapshot_info", connection).to_json(orient="records", force_ascii=False) if has_snapshot else "[]"
    digest = hashlib.sha256()
    with local.open("rb") as source:
        for chunk in iter(lambda: source.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    provenance = {"db": path.name, "sha256": digest.hexdigest(),
                  "generation_info_json": info.to_json(orient="records", force_ascii=False),
                  "snapshot_info_json": snapshot}
    return records, info, provenance

simple, simple_info, simple_provenance = read_db(SIMPLE_DB)
FLAKE_COLUMNS = [column for column in simple.columns if column.startswith("flake_")]
base = simple[KEY + SHARED + FLAKE_COLUMNS]
sources = [{**simple_provenance, "columns": ", ".join(FLAKE_COLUMNS), "renamed_from": ", ".join(FLAKE_COLUMNS)}]
qwen_columns, QUALITY_COLUMNS = {}, []

for strategy, path in QWEN_DBS.items():
    records, info, provenance = read_db(path)
    generated = info["columns"].iloc[0].split(", ")
    rename = {column: f"{strategy}_{column.rsplit('_', 1)[1]}" for column in generated}
    if sorted(rename.values()) != [f"{strategy}_en", f"{strategy}_ja"]:
        raise ValueError(f"{path.name}: expected exactly one Japanese and one English output.")
    selected_quality = []
    for column in generated:
        for suffix in ("truncated", "status", "error"):
            field = f"{column}_{suffix}"
            if field not in records:
                records[field] = None  # Unknown provenance is not asserted complete.
            rename[field] = f"{rename[column]}_{suffix}"
            selected_quality.append(field)
            QUALITY_COLUMNS.append(rename[field])
    merged = base.merge(records[KEY + SHARED + generated + selected_quality].rename(columns=rename),
                        on=KEY, how="left", suffixes=("", f"_{strategy}"), validate="one_to_one")
    for column in SHARED:
        mismatch = merged[f"{column}_{strategy}"].notna() & merged[column].ne(merged[f"{column}_{strategy}"])
        if mismatch.any():
            raise ValueError(f"{path.name}: original {column} differs in {mismatch.sum()} rows.")
    base = merged.drop(columns=[f"{column}_{strategy}" for column in SHARED])
    qwen_columns[strategy] = [f"{strategy}_ja", f"{strategy}_en"]
    sources.append({**provenance, "columns": ", ".join(qwen_columns[strategy]), "renamed_from": ", ".join(generated)})
    print(strategy, "quality flags retained from", path.name)

QWEN_COLUMNS = [column for columns in qwen_columns.values() for column in columns]
ORDER = KEY + ["text", "text_en"] + QWEN_COLUMNS + FLAKE_COLUMNS + QUALITY_COLUMNS + ["output"]
test_prompts = base[ORDER]
print(f"{len(test_prompts)} source rows; {len(ORDER)} columns including preparation quality/provenance.")


## 2. Check coverage

The Qwen columns are empty for rows a notebook hasn't processed yet (its `LIMIT`), so check how many rows have everything.

In [ ]:
PROMPT_COLUMNS = QWEN_COLUMNS + FLAKE_COLUMNS
coverage = test_prompts[PROMPT_COLUMNS].notna().sum().rename("filled").to_frame()
coverage["missing"] = len(test_prompts) - coverage["filled"]
display(coverage)
complete = test_prompts[PROMPT_COLUMNS].notna().all(axis=1)
quality_flags = [column for column in QUALITY_COLUMNS if column.endswith("_truncated")]
quality_pass = test_prompts[quality_flags].eq(False).fillna(False).all(axis=1)
print(f"{complete.sum()} rows have all prompt texts; {(complete & quality_pass).sum()} also have non-truncated rewrite flags.")
print("Unknown rewrite completion flags remain unknown; no source rows are dropped here.")


In [ ]:
# One row with every column, read top to bottom.
pd.set_option("display.max_colwidth", None)
row = complete.idxmax() if complete.any() else 0
test_prompts.loc[row].to_frame(f"row {row}")

## 3. Save as a new SQLite db (and copy it to Drive)

In [ ]:
from datetime import datetime, timezone

LOCAL_OUT = WORK_DIR / DB_OUT.name
LOCAL_OUT.unlink(missing_ok=True)
definitions = [
    f'"{column}" ' + ("INTEGER" if column.endswith("_truncated") else "TEXT")
    + (" NOT NULL" if column in KEY + ["text", "output"] else "")
    for column in ORDER
]
rows = ([None if pd.isna(value) else value for value in row] for row in test_prompts.itertuples(index=False))
with closing(sqlite3.connect(LOCAL_OUT)) as connection:
    connection.execute(f"CREATE TABLE records ({', '.join(definitions)}, PRIMARY KEY ({', '.join(KEY)}))")
    connection.executemany(f"INSERT INTO records ({', '.join(ORDER)}) VALUES ({', '.join('?' * len(ORDER))})", rows)
    pd.DataFrame([{**source, "compiled_at": datetime.now(timezone.utc).isoformat()} for source in sources]).to_sql(
        "compile_info", connection, index=False
    )
    connection.commit()
pending = DB_OUT.with_suffix(".partial")
shutil.copyfile(LOCAL_OUT, pending)
pending.replace(DB_OUT)
print("Saved new compiled dataset:", DB_OUT)
